In [4]:
import pandas as pd
from transformers import BartForConditionalGeneration, BartTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset, concatenate_datasets
import evaluate
import utils

<a name='1'></a>
## 1 - Import the Dataset

Bagian ini mempersiapkan dataset yang berasal dari dua sumber yaitu The First Certificate in English (FCE) corpus dan Birkbeck Misspelled Words

* Dataset FCE untuk Grammar Error Correction
* Dataset Birkbeck untuk Misspelled Correction

<a name='6-1'></a>
### 1.1 Grammar Error Dataset

load_dataset_from_file menghasilkan target_text dengan mengubah sebagian kata input_text berdasarkan edits(kata, index) untuk menghasilkan teks dengan grammar benar

In [4]:
train_dataset = utils.load_dataset_from_file(r"/content/dataset/dataset/fce/json/fce.train.json")
eval_dataset = utils.load_dataset_from_file(r"/content/dataset/dataset/fce/json/fce.dev.json")
test_dataset = utils.load_dataset_from_file(r"/content/dataset/dataset/fce/json/fce.test.json")

df = pd.DataFrame(train_dataset,  columns=['input_text', 'target_text'])
print(df.sample(5))

                                             input_text  \
808   Shopping is not always enjoyable.\n\nNow let's...   
1823  Dear Kim,\n\nI am very happy with your letter ...   
373    Dear Mr Robertson,\n\nI am writing you this l...   
2078  17th of June of 2000 Dear Sir, I am writing fo...   
1744  ,,Shopping is not always enjoyable". From my p...   

                                            target_text  
808   Shopping is not always enjoyable.\n\nNow let's...  
1823  Dear Kim,\n\nI am very happy with your letter ...  
373    Dear Mr Robertson,\n\nI am writing you this l...  
2078  17th of June of 2000 Dear Sir, I am writing  t...  
1744  ,,Shopping is not always enjoyable". From my p...  


<a name='6-1'></a>
### 1.2 Misspelled Dataset

make_sentence_pairs dan make_sentence_pairs1 (hanya berbeda kalimat template) menghasilkan kalimat dari kalimat template yang memiliki blank dan diisi oleh misspelled words (input_text) dan correct words (target_text)

In [5]:
file_path = r'/content/dataset/dataset/missp.dat.txt'
word_pairs = utils.read_missp_file(file_path)
sentence_pairs = utils.make_sentence_pairs(word_pairs)
sentence_pairs1 = utils.make_sentence_pairs1(word_pairs)

df_train = pd.DataFrame(sentence_pairs, columns=['input_text', 'target_text'])
df_eval = pd.DataFrame(sentence_pairs1, columns=['input_text', 'target_text'])
print(df_train.sample(5))

misspelled_train_dataset = Dataset.from_pandas(df_train)
misspelled_eval_dataset = Dataset.from_pandas(df_eval)

split_ratio = 0.2
num_eval_samples = int(len(misspelled_eval_dataset) * split_ratio)
misspelled_eval_dataset_20 = misspelled_eval_dataset.select(range(num_eval_samples))

                                          input_text  \
10779           I often misspell the word dishliged.   
31716               I often misspell the word cefes.   
21774  Could you check the spelling of negoceations?   
7942      My friend explained consisent to me today.   
11074            Have you seen the word does before?   

                                         target_text  
10779       I often misspell the word distinguished.  
31716             I often misspell the word surface.  
21774  Could you check the spelling of negotiations?  
7942     My friend explained consistent to me today.  
11074            Have you seen the word dose before?  


<a name='1'></a>
## 2 - Load BART pre-trained Model from HuggingFace

In [5]:
model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

<a name='1'></a>
## 3 - Preprocessing the Data

* Bagian ini memproses dataset dengan mengubah pasangan teks input-output (input_text, target_text) menjadi token ID yang bisa diproses oleh BART untuk sequence-to-sequence learning
* max_target_length di set ke 128 untuk mengimbangi kemampuan GPU dan membuat model lebih terlatih untuk konteks input yang lebih pendek

In [9]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=max_input_length, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["target_text"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

combined_train = concatenate_datasets([train_dataset, misspelled_train_dataset])
combined_eval = concatenate_datasets([eval_dataset, misspelled_eval_dataset_20])
tokenized_train = combined_train.map(preprocess_function, batched=True)
tokenized_eval = combined_eval.map(preprocess_function, batched=True)

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1395: FutureWarning: promote has been superseded by promote_options='default'.
  block_group = [InMemoryTable(cls._concat_blocks(list(block_group), axis=axis))]
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


Map:   0%|          | 0/38249 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3959: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/7385 [00:00<?, ? examples/s]

<a name='1'></a>
## 4 - Train the Model

In [10]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-grammar-typo-correction",
    learning_rate=4e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_dir='./logs',
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [11]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

<ipython-input-11-3182559150>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: c14220158 (c14220158-petra-christian-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.803500
1000,0.621400
1500,0.567700
2000,0.511400
2500,0.493600
3000,0.487600
3500,0.463000
4000,0.456700
4500,0.431300
5000,0.404300


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=47815, training_loss=0.2333441275876149, metrics={'train_runtime': 9755.3658, 'train_samples_per_second': 19.604, 'train_steps_per_second': 4.901, 'total_flos': 6792875002368000.0, 'train_loss': 0.2333441275876149, 'epoch': 5.0})

<a name='1'></a>
## 5 - Evaluate the Model using BLEU and ROUGE

In [10]:
model_dir = r"D:\Natural Language Processing\Project_Akhir\BART\bart-grammar-typo-correction\checkpoint-47815"

tokenizer = BartTokenizer.from_pretrained(model_dir)
model = BartForConditionalGeneration.from_pretrained(model_dir)

In [2]:
def correct_grammar(sentence: str, max_len: int = 128) -> str:
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=max_len,
        num_beams=5,
        early_stopping=True
    )
    corrected_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected_sentence

In [5]:
with open("source.txt", "r", encoding="utf-8") as f:
    data = [line.strip() for line in f]

with open("pred.txt", "w", encoding="utf-8") as pred:

    for entry in data:
        pred.write(correct_grammar(entry).replace("\n", " ") + "\n")

In [11]:
with open("pred.txt", "r", encoding="utf-8") as f:
    predictions = [line.strip() for line in f]

with open("target.txt", "r", encoding="utf-8") as f:
    targets = [line.strip() for line in f]

assert len(predictions) == len(targets), "Jumlah prediksi dan referensi harus sama!"

references = [[ref] for ref in targets]

bleu = evaluate.load("bleu")
bleu_score = bleu.compute(predictions=predictions, references=references)
print(f"\nBLEU Score: {bleu_score['bleu']:.4f}")

rouge = evaluate.load("rouge")
rouge_score = rouge.compute(predictions=predictions, references=targets)
print("\nROUGE Scores:")
for key, value in rouge_score.items():
    print(f"{key}: {value:.4f}")

Using the latest cached version of the module from C:\Users\aurum\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--bleu\9e0985c1200e367cce45605ce0ecb5ede079894e0f24f54613fca08eeb8aff76 (last modified on Fri May 16 10:31:51 2025) since it couldn't be found locally at evaluate-metric--bleu, or remotely on the Hugging Face Hub.



BLEU Score: 0.3785


Using the latest cached version of the module from C:\Users\aurum\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--rouge\b01e0accf3bd6dd24839b769a5fda24e14995071570870922c71970b3a6ed886 (last modified on Fri May 16 14:24:56 2025) since it couldn't be found locally at evaluate-metric--rouge, or remotely on the Hugging Face Hub.



ROUGE Scores:
rouge1: 0.7381
rouge2: 0.6663
rougeL: 0.7189
rougeLsum: 0.7188


<a name='1'></a>
## 6 - Try to Correct some Sentences!

In [14]:
file_path = r"D:\Natural Language Processing\Project_Akhir\dataset\input.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]

for idx, para in enumerate(paragraphs, 1):
    correction = correct_grammar(para)
    print(f"\nParagraf {idx}:")
    print("Input:")
    print(utils.wrap_text_by_words(para))
    print("\nKoreksi:")
    print(utils.wrap_text_by_words(correction))
    print("-" * 80)


Paragraf 1:
Input:
she have many friends and teacher.

Koreksi:
she has many friends and teachers.
--------------------------------------------------------------------------------

Paragraf 2:
Input:
he is a senior docter.

Koreksi:
he is a senior doctor.
--------------------------------------------------------------------------------

Paragraf 3:
Input:
Many student thinks that to learn a forein language is difficult because they
haven't enough oportunity to practise. In the school, they usualy studies
grammar, but not speak much. Also, teacher doesn’t give advices how to improve
listening skills, which make more harder to understand native speakers. Some
have tryed to watch films without subtitel, but it not helped them much.

Koreksi:
Many students think that learning a foreign language is difficult because they
haven't enough opportunities to practise. In the school, they usually study
grammar, but not speak much. Also, theteacher doesn't give advice how to improve
listening skill